# Lab 4: Online News Popularity: Will an Article Go Viral?

**Task:** predict whether an article will be shared a lot, using only what is known when it is published.

**Dataset:** Online News Popularity (UCI: https://archive.ics.uci.edu/dataset/332/online+news+popularity), included as `data/online_news_popularity.zip`, as downloaded: about 39,600 articles published by Mashable in 2013–2014, with 58 features describing each article (length, number of links and images, topic, day of publication, keyword statistics, sentiment scores) and the number of times it was shared.

**What we will do:**
1. Turn the number of shares into a yes/no "viral" target and justify the threshold.
2. Explore which features relate to popularity, and how strongly the features are correlated with each other.
3. Clean the data, transform skewed features, and rank features by correlation and mutual information.
4. Train logistic regression, a decision tree, a random forest and gradient boosting.
5. Compare them with accuracy, precision, recall, F1 and ROC AUC.
6. Report the top 10 features and what they say about what makes content go viral.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

z = zipfile.ZipFile("data/online_news_popularity.zip")
print(z.namelist())
news = pd.read_csv(z.open("OnlineNewsPopularity/OnlineNewsPopularity.csv"))
print(news.shape)
print(list(news.columns[:6]))

The column names start with a space (`' timedelta'`), an easy way to lose an hour wondering why `news["shares"]` fails. Strip them. The `.names` file in the zip describes every column; worth reading.

In [ ]:
news.columns = news.columns.str.strip()
news[["url", "timedelta", "n_tokens_title", "n_tokens_content", "num_imgs", "num_videos", "data_channel_is_tech", "weekday_is_monday", "shares"]].head()

Two columns are not features:
- `url` identifies the article (although it contains the publication date and the title words, we leave text for other labs);
- `timedelta` is the number of days between publication and the moment the data was collected. Older articles have had more time to collect shares, but this is not something we know at publication time, which is when the prediction would be made. We drop it.

## 1. Defining "viral"

In [ ]:
print(news["shares"].describe().round(0))
print(news["shares"].quantile([0.5, 0.75, 0.8, 0.9, 0.95, 0.99]))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(news["shares"], bins=100); axes[0].set_title("shares")
axes[1].hist(np.log10(news["shares"]), bins=100); axes[1].set_title("log10(shares)")
for q in [0.5, 0.9]:
    axes[1].axvline(np.log10(news["shares"].quantile(q)), color="crimson", ls="--")
plt.show()

Shares are extremely skewed: half the articles get fewer than 1,400 shares, a few get hundreds of thousands. The **median** (1,400) would split the articles into "more popular than average" and "less", a balanced problem, but half of all articles can hardly be called *viral*. We define **viral as the top 10%: 6,200 shares or more.** It matches the meaning of the word, and it's a decision a newsroom might care about (which articles to promote). The price is an imbalanced target: 10% positives, so accuracy alone will be misleading. (Try the median version at the end of the lab.)

In [ ]:
threshold = news["shares"].quantile(0.9)
news["viral"] = (news["shares"] >= threshold).astype(int)
print(f"threshold {threshold:.0f} shares -> viral share {news['viral'].mean():.3f}")

## 2. Exploratory analysis

### Data problems

In [ ]:
print("missing values:", news.isna().sum().sum())
print("articles with 0 words of content:", (news["n_tokens_content"] == 0).sum())
print("n_unique_tokens > 1 (a share can't exceed 1):", (news["n_unique_tokens"] > 1).sum())
print(news.loc[news["n_unique_tokens"] > 1, ["n_tokens_content", "n_unique_tokens", "n_non_stop_words", "n_non_stop_unique_tokens"]])
print("negative values in keyword statistics:", (news.filter(like="kw_") < 0).sum()[lambda s: s > 0].to_dict())

- 1,181 articles have no content words: they are probably galleries or video pages, so their text statistics (rates, sentiments) are all 0. They are real articles; we keep them and add an indicator `no_text` so the models can treat them separately.
- One article has impossible ratios (n_unique_tokens = 701, a share of words that must lie between 0 and 1): a data error. We drop that row.
- `kw_min_min` and a few other keyword statistics use −1 for "not available". Tree models handle such codes easily; for logistic regression it would be better to add an indicator, but these are minor features, so we leave them.

In [ ]:
news = news[news["n_unique_tokens"] <= 1].copy()
news["no_text"] = (news["n_tokens_content"] == 0).astype(int)
y = news["viral"]
X = news.drop(columns=["url", "timedelta", "shares", "viral"])
print(X.shape)

### Which features relate to popularity?

We look at the correlation of each feature with **log(shares)**, which is less dominated by a few huge values than the raw counts.

In [ ]:
log_shares = np.log(news["shares"])
corr_target = X.corrwith(log_shares).sort_values()
fig, ax = plt.subplots(figsize=(6, 12))
corr_target.plot.barh(ax=ax); ax.set_title("correlation with log(shares)")
plt.show()
print(corr_target.round(3).tail(8)); print(corr_target.round(3).head(8))

All correlations are weak (the largest is about 0.2): popularity is hard to predict from these features, and we shouldn't expect high accuracy. The strongest positive ones are the keyword statistics (`kw_avg_avg`: how many shares articles with the same keywords got on average), self-references (shares of other Mashable articles that this one links to), and some topics; the entertainment and world channels go with fewer shares.

### Multicollinearity

In [ ]:
corr = X.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
plt.xticks(fontsize=6); plt.yticks(fontsize=6); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(pairs[pairs.abs() > 0.8].sort_values(key=abs, ascending=False).round(3))

Several groups of features say almost the same thing: the ratios of unique and non-stop words, the min/max/average self-reference shares, the keyword statistics, and the world-news channel with the LDA topic score that detects world news from the text. And one surprise: our new `no_text` indicator has correlation $-1$ with `n_non_stop_words`, which turns out to be 1 for every article with text and 0 for the others; it was already an indicator. Strong correlations among features make individual logistic regression weights unstable and hard to interpret (multicollinearity), although they don't hurt predictions much; tree models are not affected in the same way. For logistic regression we drop one feature from each very highly correlated pair (|r| > 0.9).

In [ ]:
to_drop = set()
for (a, b), r in pairs[pairs.abs() > 0.9].items():
    if a not in to_drop and b not in to_drop:
        to_drop.add(b)
print("dropped for logistic regression:", sorted(to_drop))

## 3. Preprocessing

### Skewed features
Many count features (number of words, links, images, keyword share statistics) are heavily right-skewed. A few extreme articles then dominate a linear model's fit. A log transform, $\log(1 + x)$, pulls in the long tail; we apply it to non-negative features with skewness above 2. Trees don't need it: they only care about the order of values.

In [ ]:
skew = X.skew()
skewed = [c for c in X.columns if skew[c] > 2 and X[c].min() >= 0 and X[c].nunique() > 2]
print(len(skewed), "skewed features, e.g.", skewed[:6])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(X["num_hrefs"], bins=50); axes[0].set_title("num_hrefs")
axes[1].hist(np.log1p(X["num_hrefs"]), bins=50); axes[1].set_title("log(1 + num_hrefs)")
plt.show()

### Train/test split, then feature ranking
Split first (stratified, so both sets have 10% viral articles); then everything learned from data (rankings, transformations, models) uses the training set only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
mi = pd.Series(mutual_info_classif(x_train, y_train, random_state=0), index=X.columns)
ranking = pd.DataFrame({"|corr| with viral": x_train.corrwith(y_train).abs(), "mutual information": mi})
ranking = ranking.sort_values("mutual information", ascending=False)
ranking.head(15).round(4)

**Mutual information** measures any kind of dependence between a feature and the target (not only straight-line relationships, as correlation does). Both rankings put the keyword statistics, self-references, the LDA topic scores and the data channels near the top. Does selecting only the top-ranked features help? We'll compare logistic regression on all features with the top 20 by mutual information.

## 4. Models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

def logistic_pipe(features):
    sk = [f for f in features if f in skewed]
    rest = [f for f in features if f not in skewed]
    prep = ColumnTransformer([("log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("scale", StandardScaler())]), sk),
                              ("scale", StandardScaler(), rest)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=5000, class_weight="balanced", C=0.1))])

results, fitted, test_scores = [], {}, {}
def evaluate(name, model, features):
    model.fit(x_train[features], y_train)
    p = model.predict_proba(x_test[features])[:, 1]
    pred = (p >= 0.5).astype(int)
    fitted[name], test_scores[name] = (model, features), p
    results.append({"model": name, "accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
                    "recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred), "ROC AUC": roc_auc_score(y_test, p)})

lr_features = [c for c in X.columns if c not in to_drop]
top20 = ranking.index[:20].tolist()
evaluate("logistic (all features)", logistic_pipe(lr_features), lr_features)
evaluate("logistic (top 20 by MI)", logistic_pipe([f for f in top20 if f in lr_features]), [f for f in top20 if f in lr_features])
pd.DataFrame(results).round(3)

We use `class_weight="balanced"` so that the 10% viral articles count as much as the other 90% in the loss; without it, a cutoff of 0.5 would flag almost nothing. The 20 top-ranked features lose a little AUC compared with all 55 (0.695 against 0.710) and do just as well when flagging the top 10% (below): most of the signal is in a few feature groups.

### Trees and ensembles
Tree depth and leaf size are tuned with 3-fold cross-validation on the training data, scoring by ROC AUC.

In [ ]:
features = X.columns.tolist()
tree = GridSearchCV(DecisionTreeClassifier(class_weight="balanced", random_state=0),
                    {"max_depth": [4, 6, 8, 12], "min_samples_leaf": [20, 50, 100]}, cv=3, scoring="roc_auc", n_jobs=-1)
evaluate("decision tree", tree, features)
print("tree:", tree.best_params_)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.3, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=0)
evaluate("random forest", rf, features)

gbm = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0)
evaluate("gradient boosting", gbm, features)
pd.DataFrame(results).round(3)

## 5. Evaluation

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, p in test_scores.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.legend(fontsize=8)
plt.show()

Reading the table:
- **Accuracy** is not a good guide here: the random forest and gradient boosting predict "viral" so rarely at a cutoff of 0.5 that their accuracy is high (about 0.9, the share of non-viral articles) and their recall tiny; the class-weighted logistic regression and tree flag many more articles and have lower accuracy. **Precision, recall and F1 depend on the cutoff**, which can be chosen for the purpose (for example, the top 5% of predicted articles get promoted).
- **ROC AUC** compares the ranking over all cutoffs: the ensembles rank articles best, then logistic regression, and a single tree is clearly weaker.
- All AUCs are modest (around 0.7). As the correlations suggested, popularity is only partly predictable from these features: much depends on the content itself, timing and luck.

For a fair comparison at one operating point, here is precision and recall when each model flags its top 10% of articles:

In [ ]:
rows = []
for name, p in test_scores.items():
    flag = p >= np.quantile(p, 0.9)
    rows.append({"model": name, "precision (top 10% flagged)": precision_score(y_test, flag), "recall": recall_score(y_test, flag)})
pd.DataFrame(rows).round(3)

## 6. The top 10 influential features

**Tree importance** (random forest, mean decrease in impurity) and the **logistic regression weights** (on standardised, log-transformed features, so comparable: the change in log-odds of going viral per standard deviation).

In [ ]:
rf_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("random forest importance:\n", rf_imp.head(10).round(4))

lr_model, lr_feats = fitted["logistic (all features)"]
names = [n.split("__")[1] for n in lr_model[0].get_feature_names_out()]
weights = pd.Series(lr_model[-1].coef_[0], index=names)
print("\nlogistic regression weights (largest in absolute value):\n", weights.reindex(weights.abs().sort_values(ascending=False).index).head(10).round(3))

Notice the logistic weights of the self-reference features: +0.82 for the average, −0.40 for the maximum. These three features are strongly correlated, and the model balances them against each other: the classic symptom of multicollinearity. Individually such weights shouldn't be read literally; together they say "popular self-references go with viral articles".

Impurity-based importance favours features with many distinct values, so a permutation importance on the test data is a useful check:

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, x_test, y_test, scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1)
pd.Series(perm.importances_mean, index=features).sort_values(ascending=False).head(10).round(4)

### What makes content go viral? (interpretation)

All three views agree on the main points:
1. **Keywords with a track record.** The keyword statistics (`kw_avg_avg`, `kw_max_avg`: how many shares earlier articles with the same keywords got) are the strongest predictors. Topics that went viral before tend to go viral again.
2. **Popular neighbours.** Articles linking to other Mashable articles that were widely shared (`self_reference_*`) are more often viral: a sign of being part of a popular story or series.
3. **Topic.** The LDA topic scores (how much an article belongs to each of five topics found automatically in the text) matter: topic 3 goes with more sharing, topic 2 (which mostly matches the world-news channel) with less; world news and entertainment are less often viral.
4. **Weekend publishing** helps a little (correlation 0.11 with log shares): fewer articles compete for attention.
5. **Links and images** have small positive effects; sentiment and length matter much less than one might expect.

These are associations in historical data, not causes: an editor can't make an article viral just by choosing popular keywords, since the keyword effect partly reflects which stories were already big news. And the models' modest AUC (about 0.7) is itself a finding: most of what makes an article go viral is not captured by these features.

**Try:** define viral as shares above the median (balanced classes) and compare the metrics; tune the gradient boosting model (learning rate, number of trees with early stopping).